# 02 — Preprocessing & Labeling

**Goal:** Clean the raw DataFrame (NaN policy, column selection), compute 1-day forward returns, derive a data-driven threshold from training data only, and produce Bull / Bear / Neutral labels.

**Modules used:** `src/data/preprocess.py`, `src/data/labeling.py`

---

## 0 · Imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data.load_data import load_market_data
from src.data.preprocess import apply_missing_value_policy, select_columns
from src.data.labeling import (
    compute_forward_return,
    compute_threshold,
    make_labels,
    summarize_label_distribution,
)

DATA_PATH = ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv'

FEATURE_COLS = [
    'VIX_Term_Structure', 'Yield_Curve', 'SKEW_Index',
    'Risk_Appetite_Ratio', 'Crude_Oil', 'VWAP_Deviation', 'Volume_Momentum'
]
DATE_COL  = 'Date'
CLOSE_COL = 'Nasdaq_Close'

---
## 1 · Load raw data

In [ ]:
df_raw = load_market_data(str(DATA_PATH), date_col=DATE_COL)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

---
## 2 · Missing value policy: `ffill_then_drop_head`

**Strategy:**
1. **Forward-fill** — each NaN inherits the last known value (e.g. holiday closes).
2. **Drop head** — any row still NaN after ffill (i.e. NaN at the very start of the series) is removed.

We never back-fill because that would introduce future information into past rows.

In [ ]:
missing_before = df_raw[FEATURE_COLS + [CLOSE_COL]].isnull().sum().sum()
print(f'Missing cells before policy: {missing_before}')

df_clean = apply_missing_value_policy(df_raw, method='ffill_then_drop_head')

missing_after = df_clean[FEATURE_COLS + [CLOSE_COL]].isnull().sum().sum()
print(f'Missing cells after  policy: {missing_after}')
print(f'Rows removed: {len(df_raw) - len(df_clean)}')

---
## 3 · Select relevant columns

In [ ]:
df = select_columns(df_clean, feature_cols=FEATURE_COLS, close_col=CLOSE_COL, date_col=DATE_COL)
print(f'Selected shape: {df.shape}')
print('Columns:', df.columns.tolist())

---
## 4 · Forward return computation

$$\text{forward\_return}_t = \frac{\text{Close}_{t+1}}{\text{Close}_t} - 1$$

The last row has no t+1 counterpart, so it becomes `NaN` and will be labeled **Neutral** (and dropped later).

In [ ]:
forward_returns = compute_forward_return(df[CLOSE_COL], horizon=1)

print(f'Return series length : {len(forward_returns)}')
print(f'NaN count            : {forward_returns.isna().sum()}  (last row, no t+1 available)')
print(f'Mean  return         : {forward_returns.mean():.4%}')
print(f'Std   return         : {forward_returns.std():.4%}')
forward_returns.describe()

### Distribution of daily returns

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(forward_returns.dropna(), bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Distribution of 1-Day Forward Returns', fontsize=12)
ax.set_xlabel('Return')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

---
## 5 · Threshold computation

**Critical rule:** The threshold is computed **only from training returns**. We simulate what the pipeline does inside each CV fold by computing it on the first 85% of the data (the development set).

**Method:** `quantile` — `threshold = quantile(|train_returns|, q=0.40)`

Any move smaller than ±threshold is considered noise (Neutral) and dropped.

In [ ]:
# Approximate dev set for illustration (full pipeline uses fold-level splits)
n_dev = int(len(forward_returns) * 0.85)
train_returns_demo = forward_returns.iloc[:n_dev]

threshold = compute_threshold(train_returns_demo, method='quantile', quantile=0.40)

print(f'Threshold (q=0.40 of |train returns|): {threshold:.4%}')
print(f'→  Bull  if return >  {threshold:.4%}')
print(f'→  Bear  if return < -{threshold:.4%}')
print(f'→  Neutral otherwise  (|return| <= {threshold:.4%})')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(forward_returns.dropna(), bins=80, color='lightgray', edgecolor='white', linewidth=0.3, label='All returns')
ax.axvline( threshold, color='green', linewidth=1.5, label=f'+threshold ({threshold:.3%})')
ax.axvline(-threshold, color='red',   linewidth=1.5, label=f'-threshold (-{threshold:.3%})')
ax.axvspan(-threshold, threshold, alpha=0.15, color='yellow', label='Neutral zone')
ax.set_title('Forward Returns with Bull / Bear Threshold', fontsize=12)
ax.set_xlabel('Return')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · Label generation

In [ ]:
labels = make_labels(forward_returns, threshold=threshold,
                     bull_label=1, bear_label=0, neutral_label=-1)

dist = summarize_label_distribution(labels)
label_names = {1: 'Bull', 0: 'Bear', -1: 'Neutral'}
dist.index = dist.index.map(lambda x: f'{label_names.get(x, x)} ({x})')
print(dist.to_string())

### Label distribution — bar chart

In [ ]:
counts = labels.value_counts().sort_index()
names  = {-1: 'Neutral\n(-1)', 0: 'Bear\n(0)', 1: 'Bull\n(1)'}
colors = ['gold', 'tomato', 'mediumseagreen']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([names[k] for k in counts.index], counts.values,
               color=[colors[i] for i in range(len(counts))], edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=10)
ax.set_title('Label Distribution (before neutral removal)', fontsize=12)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### Labels over time

In [ ]:
dates = df[DATE_COL]
color_map = {1: 'green', 0: 'red', -1: 'gold'}
label_color = labels.map(color_map)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(dates, df[CLOSE_COL], linewidth=0.8, color='steelblue')
axes[0].set_ylabel('Nasdaq Close')
axes[0].set_title('Price & Label Overlay')

axes[1].scatter(dates, labels, c=label_color, s=2, alpha=0.6)
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['Neutral', 'Bear', 'Bull'])
axes[1].set_ylabel('Label')

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

---
## 7 · After neutral removal

Neutral samples are dropped **after** labeling — never before. This ensures the threshold is computed on the full return distribution.

In [ ]:
active_mask = labels != -1
active_labels = labels[active_mask]

bull_pct = (active_labels == 1).mean() * 100
bear_pct = (active_labels == 0).mean() * 100

print(f'Samples after neutral removal : {active_mask.sum()} / {len(labels)}')
print(f'Bull : {bull_pct:.1f}%')
print(f'Bear : {bear_pct:.1f}%')
print(f'Imbalance ratio (bull/bear)   : {bull_pct/bear_pct:.3f}')

---
## Summary

| Step | Output |
|---|---|
| Missing value policy | Forward-fill → drop head rows |
| Forward return horizon | 1 trading day |
| Threshold method | 40th percentile of `|train returns|` |
| Label encoding | Bull = 1, Bear = 0, Neutral = -1 |
| Neutral policy | Drop after labeling |

**Next:** `03_splitting_and_sequences.ipynb` — chronological dev/test split and sliding-window sequence construction.